# TTS Voice Conversion

Converts every clip in a manifest to one or more named voice identities with
[voiceclonnx](https://github.com/TigreGotico/voiceclonnx). This is step 6 of
the [TTS dataset pipeline](../docs/pipeline.md): it reads a manifest (see
[manifest.md](../docs/manifest.md)) -- the output of donor synthesis, corpus
ingest or enhancement -- and an optional selection file, and writes one new
manifest per identity holding the converted audio. Each output row keeps its
parent's `id` so a later step can trace a converted clip back to its source.

This notebook does not score a clip and does not decide what a trainer keeps.
The measure notebook that follows scores the converted audio, and the select
notebook after that decides what stays; see [pipeline.md](../docs/pipeline.md)
for the full chain and [registry.md](../docs/registry.md) for where the donor
or corpus behind the input manifest was admitted.

### Pipeline stages

| # | Stage | Description |
|---|---|---|
| 0 | Configuration | Set all paths, identities and engine choice via env vars |
| 1 | Install | Fetch voiceclonnx and the onnxruntime build named by `ONNX_RUNTIME` |
| 2 | Load manifest | Read the input manifest and apply a selection file if one is given (or, with `SMOKE=True` and no manifest, build a small one with its own reference voices) |
| 3 | Identities and cloner | Resolve one reference WAV per identity, build the `VoiceCloner` |
| 4 | Voice conversion | Convert every row to every identity, write one manifest per identity |
| 5 | Output summary | Row counts per identity, and the hard checks this notebook enforces |

### Output structure
```
VC_OUTPUT_DIR/
  miro/
    wavs/
      row1-miro.wav
      ...
    miro.jsonl
    miro.jsonl.meta.json
  dii/
    wavs/
      row1-dii.wav
      ...
    dii.jsonl
    dii.jsonl.meta.json
```

### Usage
1. Edit the configuration cell below: point `VC_MANIFEST` at your input
   manifest, and set one `REFERENCE_<IDENTITY>` WAV per name in `IDENTITIES`.
2. Run cells sequentially.
3. Existing output WAVs are left alone on a re-run (resume-safe); the output
   manifest is always rewritten in full so its row count still matches the
   input.

> **Credits:** Voice conversion is [voiceclonnx](https://github.com/TigreGotico/voiceclonnx),
> a pure-onnxruntime multi-engine library with no torch at runtime. The
> SMOKE path uses the `triaan` engine at int8 (about 84 MB total), so the
> smoke run needs no multi-hundred-MB download.

> This notebook was written with AI assistance (Claude, via Claude Code) and has not been reviewed by a human. Verify outputs before relying on them.

> This project was funded through the NGI0 Commons Fund, a fund established by NLnet with financial support from the European Commission's Next Generation Internet programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No 101135429. Additional funding is made available by the Swiss State Secretariat for Education, Research and Innovation (SERI).

| | |
|---|---|
| **Inputs** | `VC_MANIFEST`: a JSON Lines manifest (docs/manifest.md); optional `VC_SELECTION`; one `REFERENCE_<IDENTITY>` WAV per name in `IDENTITIES`. With `SMOKE=True` and no manifest at `VC_MANIFEST`, the notebook builds one from up to `N_SAMPLES` short edge-tts clips, plus a demo reference WAV for every identity with none set. |
| **Outputs** | one manifest per identity (`VC_OUTPUT_DIR/<identity>/<identity>.jsonl`), its `.meta.json` sidecar, and the converted WAVs under `<identity>/wavs/`. |
| **Runtime** | SMOKE (triaan int8, a couple of demo clips): well under a minute on CPU including the model download. Full run with the default `chatterbox` engine: a few seconds per clip once its weights are cached, plus the one-time download below. |
| **Hardware** | CPU only; every engine that accepts an onnxruntime provider list gets one from `ONNX_PROVIDERS`/`ONNX_RUNTIME`. No GPU is required. |
| **Cost** | No paid compute or API calls. `triaan` at int8 (the SMOKE engine) is about 84 MB; `chatterbox` (the full-run default) downloads roughly 470 MB (quantized) to 1.1 GB (fp32) of free public ONNX weights from Hugging Face Hub on first use. |

CI: skipped by this repository's execute job (`ngi_deliverable.skip_execute=true`: the full-run default engine is a several-hundred-MB download); validated only by the lint job (nbformat + nbstripout), which does not parse or compile code cells. SMOKE's `triaan` int8 engine is light enough to run in that job if this notebook is ever added to it.


In [ ]:
# Parameters
SMOKE = False    # True: VC_ENGINE defaults to the tiny triaan engine (int8), and a built-in demo
                 # manifest plus demo reference voices are built when VC_MANIFEST does not exist
N_SAMPLES = 32   # real manifest: rows are capped to this many; demo: this many clips are built (max 2)


---
## 0 · Global Configuration

All parameters are set as environment variables via `os.environ.setdefault()`.
Override any of them from your shell, a `.env` file, or `%env` magic **before**
running the config cell.

**Edit the values below to match your setup, then run the cell.**

| Env var | Meaning |
|---|---|
| `VC_MANIFEST` | Path to the input manifest: JSON Lines, [manifest.md](../docs/manifest.md) |
| `VC_SELECTION` | Path to a selection file (JSON Lines of kept `id` values); empty converts every row |
| `IDENTITIES` | Comma list of voice identities to convert to |
| `REFERENCE_<IDENTITY>` | Reference WAV for that identity (uppercased), e.g. `REFERENCE_MIRO`, `REFERENCE_DII`; required for every name in `IDENTITIES` |
| `VC_OUTPUT_DIR` | Base output directory; one subdirectory per identity is written under it |
| `VC_ENGINE` | voiceclonnx engine alias (default `chatterbox`); `voiceclonnx list` prints the rest |
| `VC_<OPTION>` | One env var per constructor option of the selected engine — `VC_QUANTIZED`, `VC_EXAGGERATION`, `VC_ODE_STEPS`, `VC_CHUNK_SAMPLES`, `VC_K`, `VC_DEFAULT_MODEL`, `VC_SAMPLE_RATE`, `VC_SPEAKER_ID`, `VC_F0_UP_KEY`; empty means the engine's own default. The next cell lists every option every engine takes; setting one an engine does not accept stops the notebook, naming the engine and the option. |
| `ONNX_PROVIDERS` | onnxruntime execution providers: a comma list, or `auto`; passed only to the voiceclonnx engines whose adapter accepts a provider list |
| `ONNX_RUNTIME` | Which onnxruntime build the install cell fetches: `cpu`, `cuda`, `rocm` |

In [ ]:
import os


def _default_base_dir(name):
    """Somewhere writable that survives the session on the current platform."""
    if os.path.isdir("/kaggle"):
        return f"/kaggle/working/{name}"
    if os.path.isdir("/content"):
        return f"/content/{name}"
    return f"./{name}"


def _env(key, default=""):
    return os.environ.get(key, default)


# ──────────────────────────────────────────────
# INPUT  (Stage 2)
# ──────────────────────────────────────────────
os.environ.setdefault("VC_MANIFEST",   "manifest.jsonl")   # Input manifest: JSON Lines, docs/manifest.md
os.environ.setdefault("VC_SELECTION",  "")                 # Selection file (JSON Lines of kept ids); empty = every row

# ──────────────────────────────────────────────
# VOICE IDENTITIES  (Stages 3, 4)
#   One reference WAV per name in IDENTITIES, from an explicit REFERENCE_<NAME>
#   variable. This is a named argument per identity, never a language-to-engine
#   default: a missing REFERENCE_<NAME> stops the notebook. (RVC is the one
#   engine where this value names a voice model, not a WAV — see the option
#   tables in the next cell.)
# ──────────────────────────────────────────────
os.environ.setdefault("IDENTITIES",     "miro,dii")        # Comma list of voice identities to produce
os.environ.setdefault("REFERENCE_MIRO", "")                # Reference WAV for the "miro" identity
os.environ.setdefault("REFERENCE_DII",  "")                # Reference WAV for the "dii" identity

os.environ.setdefault("VC_OUTPUT_DIR",  _default_base_dir("voice_convert_output"))  # One subdirectory per identity is written under this

# ──────────────────────────────────────────────
# VOICE CONVERSION ENGINE  (Stage 4)
#   voiceclonnx.VoiceCloner(engine=VC_ENGINE, **kwargs). One env var per
#   constructor option of the SELECTED engine, named VC_<OPTION>; an empty
#   value means "use the engine's own default". Setting an option the selected
#   engine does not accept stops the notebook (Stage 3), naming the engine and
#   the option. `voiceclonnx list` (CLI) prints every registered engine; the
#   next cell documents every option every engine takes.
# ──────────────────────────────────────────────
os.environ.setdefault("VC_ENGINE", "triaan" if SMOKE else "chatterbox")   # voiceclonnx engine alias; SMOKE defaults to the tiny triaan engine

os.environ.setdefault("VC_QUANTIZED",     "1" if SMOKE else "")   # bool  — chatterbox, cosyvoice, bicodec, facodec, focalcodec, knnvc, lscodec, openvoice, rvc, triaan; SMOKE uses int8 weights
os.environ.setdefault("VC_EXAGGERATION",  "")   # float — chatterbox only
os.environ.setdefault("VC_ODE_STEPS",     "")   # int   — cosyvoice only
os.environ.setdefault("VC_CHUNK_SAMPLES", "")   # int   — bicodec only
os.environ.setdefault("VC_K",             "")   # int   — focalcodec, knnvc
os.environ.setdefault("VC_DEFAULT_MODEL", "")   # str   — rvc only
os.environ.setdefault("VC_SAMPLE_RATE",   "")   # int   — rvc only
os.environ.setdefault("VC_SPEAKER_ID",    "")   # int   — rvc only
os.environ.setdefault("VC_F0_UP_KEY",     "")   # int   — rvc only

# ──────────────────────────────────────────────
# ONNX RUNTIME  (Stage 1, 3)
#   Stage 3 reads the chosen engine's own adapter signature to decide whether
#   it takes a `providers` argument at all — it does not hardcode which engine
#   that is, since a future voiceclonnx release can add one.
# ──────────────────────────────────────────────
os.environ.setdefault("ONNX_PROVIDERS", "auto")            # comma list of onnxruntime providers, or "auto"
os.environ.setdefault("ONNX_RUNTIME",   "cpu")              # cpu | cuda | rocm — which onnxruntime build the install cell fetches

_set_vc_options = {
    k: v for k, v in sorted(os.environ.items())
    if k.startswith("VC_") and k not in ("VC_ENGINE", "VC_MANIFEST", "VC_SELECTION", "VC_OUTPUT_DIR") and v.strip()
}

print("Configuration loaded.")
print(f"  Input manifest : {_env('VC_MANIFEST')}")
print(f"  Selection      : {_env('VC_SELECTION') or '(none — every row is converted)'}")
print(f"  Identities     : {_env('IDENTITIES')}")
print(f"  Output dir     : {_env('VC_OUTPUT_DIR')}")
print(f"  VC engine      : {_env('VC_ENGINE')}")
print(f"  VC options set : {_set_vc_options or '(none — engine defaults)'}")
print(f"  ONNX providers : {_env('ONNX_PROVIDERS')}  (onnxruntime build: {_env('ONNX_RUNTIME')})")

---
## 0a · Voice Conversion Engine Options

Every `voiceclonnx` 0.0.3a3 engine's constructor is read through
`inspect.signature` before use (Stage 3): only the options in the table for
`VC_ENGINE` are read from the environment, and setting a `VC_<OPTION>` an
engine does not accept stops the notebook, naming the engine and the option.
`providers` is not one of these — every engine that accepts an onnxruntime
provider list gets it from `ONNX_PROVIDERS` (above), never from a `VC_`
variable, because it is resolved once, the same way, for every engine that
takes it.

**chatterbox** (`VC_ENGINE=chatterbox`, default)

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load the INT8 weights (~468 MB) instead of fp32 (~1081 MB); same WER on reference clips. |
| `exaggeration` | `VC_EXAGGERATION` | `0.6` | Voice-exaggeration scalar. Stored for API compatibility only: the decoder has no direct input for it, since it is baked into the speaker embeddings at training time. |

**cosyvoice**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models. |
| `ode_steps` | `VC_ODE_STEPS` | `10` | Euler ODE steps for the flow decoder; more steps trade speed for quality. |

**bicodec**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models. |
| `chunk_samples` | `VC_CHUNK_SAMPLES` | `32000` | Wav2Vec2 chunking window in samples (32000 = 2 s at 16 kHz). Lower it on out-of-memory; `0` disables chunking and processes the full clip at once. |

**facodec**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 models; default is fp32, for quality. |

**focalcodec**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models. Not recommended for production: 31% WER at int8 against 15% at fp32. |
| `k` | `VC_K` | `4` | Nearest neighbours for the cosine-distance kNN matching step. |

**knnvc**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models; fp32 is the default for quality, int8 trades quality for faster CPU inference. |
| `k` | `VC_K` | `4` | Nearest neighbours (L2 distance) for the matching step. |

**lscodec**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models; fp32 is the supported quality path. |

**openvoice**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models. |

**rvc**

`REFERENCE_<IDENTITY>` names an RVC `.onnx` voice model for this engine — a
local path or an HF repo ID — never a reference WAV: RVC bakes the target
speaker into the model weights, and reference audio is never read.

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Use INT8 ContentVec/RMVPE base models. Not recommended for production: 69% WER at int8 against 38% at fp32. |
| `default_model` | `VC_DEFAULT_MODEL` | (none) | Fallback RVC voice model (path or HF repo ID), used only when a `REFERENCE_<IDENTITY>` value is empty. |
| `sample_rate` | `VC_SAMPLE_RATE` | `40000` | Output sample rate override; the adapter otherwise reads it from the synthesizer model's own metadata. |
| `speaker_id` | `VC_SPEAKER_ID` | `0` | Speaker index for multi-speaker synthesizer models. |
| `f0_up_key` | `VC_F0_UP_KEY` | `0` | Pitch shift in semitones applied to the extracted F0 before synthesis. |

**triaan**

| Option | Env var | Engine default | Meaning |
|---|---|---|---|
| `quantized` | `VC_QUANTIZED` | `false` | Load INT8 quantized ONNX models. |

---
## 1 · Install

| Env var | |
|---|---|
| `ONNX_RUNTIME` | `cpu` installs `onnxruntime`; `cuda` installs `onnxruntime-gpu`; `rocm` prints AMD's own install instructions and nothing is installed here |

In [ ]:
onnx_runtime = _env("ONNX_RUNTIME", "cpu").strip().lower()

%pip install --quiet --pre "voiceclonnx>=0.0.3a3" "tqdm>=4.70.1"

if onnx_runtime == "cuda":
    %pip install --quiet --pre "onnxruntime-gpu>=1.30.0"
elif onnx_runtime == "rocm":
    print("AMD ROCm: install onnxruntime-rocm or the MIGraphX build from AMD's package "
          "index yourself; nothing is installed here and CPU is the fallback.")
else:
    %pip install --quiet --pre "onnxruntime>=1.30.0"

---
## 2 · Load Manifest

Reads the input manifest and, when `VC_SELECTION` names one, keeps only the
rows whose `id` is in the selection file (JSON Lines of kept `id` values,
[manifest.md](../docs/manifest.md)). Audio paths in the manifest are relative
to the manifest's own directory. With `SMOKE=True` and no manifest at `VC_MANIFEST`, this stage builds a small demo manifest and, for any identity in `IDENTITIES` with no `REFERENCE_<IDENTITY>` set, a demo reference WAV of its own.

In [ ]:
import json
from pathlib import Path


def _load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _load_selection(path):
    """A selection file: one kept `id` per line (docs/manifest.md). None keeps every row."""
    if not path:
        return None
    ids = set()
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            ids.add(obj["id"] if isinstance(obj, dict) else obj)
    return ids


manifest_path = Path(_env("VC_MANIFEST"))
if not manifest_path.is_file():
    if not SMOKE:
        raise RuntimeError(
            f"VC_MANIFEST={manifest_path} does not name a file. Set it to the manifest this "
            f"notebook should convert (docs/manifest.md), or set SMOKE=True for the built-in demo."
        )
    # SMOKE's built-in demo: this notebook has no default manifest of its own
    # (it only converts audio another step already produced), so the smoke
    # path builds a couple of short edge-tts clips, plus one demo reference
    # WAV per identity that has none configured.
    %pip install --quiet --pre "edge-tts>=7.2.8" "nest-asyncio>=1.6.0" "soundfile>=0.14.0"
    import asyncio
    import subprocess

    import nest_asyncio
    nest_asyncio.apply()
    import edge_tts
    import soundfile as _sf

    demo_dir = Path(_default_base_dir("tts_voice_convert_demo"))
    demo_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = demo_dir / "manifest.jsonl"

    async def _demo_synth(text, voice, out_mp3):
        await edge_tts.Communicate(text, voice).save(str(out_mp3))

    def _demo_clip(name, text, voice):
        mp3_path = demo_dir / f"{name}.mp3"
        wav_path = demo_dir / f"{name}.wav"
        asyncio.run(_demo_synth(text, voice, mp3_path))
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp3_path), "-ar", "16000", "-ac", "1", str(wav_path)],
            check=True,
        )
        mp3_path.unlink()
        info = _sf.info(str(wav_path))
        return wav_path.name, round(info.frames / info.samplerate, 3), info.samplerate

    DEMO_SENTENCES = [
        "Hey Mycroft, what is the weather like today in Lisbon?",
        "Set a timer for ten minutes.",
    ][:max(1, min(int(N_SAMPLES), 2))]
    DEMO_SRC_VOICE = "en-GB-SoniaNeural"

    demo_rows = []
    for i, text in enumerate(DEMO_SENTENCES, 1):
        clip_id = f"demo_{i:04d}"
        name, dur, sr = _demo_clip(clip_id, text, DEMO_SRC_VOICE)
        demo_rows.append({
            "id": clip_id, "text": text, "lang": "en-GB", "audio": name,
            "duration_s": dur, "sample_rate": sr,
            "source": {"kind": "tts", "plugin": "edge-tts", "voice": DEMO_SRC_VOICE},
        })
    with open(manifest_path, "w", encoding="utf-8") as f:
        for row in demo_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"SMOKE=True and VC_MANIFEST does not exist: wrote a {len(demo_rows)}-row demo manifest to {manifest_path}")

    DEMO_IDENTITY_VOICES = {"miro": "en-US-GuyNeural", "dii": "en-US-JennyNeural"}
    for _identity in [i.strip() for i in _env("IDENTITIES", "miro,dii").split(",") if i.strip()]:
        _env_name = f"REFERENCE_{_identity.upper()}"
        if os.environ.get(_env_name, "").strip():
            continue
        _voice = DEMO_IDENTITY_VOICES.get(_identity, DEMO_SRC_VOICE)
        _ref_name, _ref_dur, _ref_sr = _demo_clip(f"reference_{_identity}", "This is a short reference voice sample.", _voice)
        os.environ[_env_name] = str(demo_dir / _ref_name)
        print(f"  {_env_name} not set: wrote a demo reference to {os.environ[_env_name]}")

manifest_dir = manifest_path.parent

all_rows = _load_jsonl(str(manifest_path))
selection = _load_selection(_env("VC_SELECTION"))
input_rows = all_rows if selection is None else [r for r in all_rows if r["id"] in selection]
if SMOKE:
    input_rows = input_rows[:N_SAMPLES]
    print(f"SMOKE=True: capped to {len(input_rows)} row(s).")

if not input_rows:
    raise RuntimeError(
        "No input rows to convert: the manifest is empty, or the selection kept none of its rows."
    )

print(f"Manifest: {len(all_rows)} rows in {manifest_path}")
if selection is not None:
    print(f"Selection: {len(selection)} ids kept -> {len(input_rows)} rows to convert")
else:
    print(f"No selection file: converting every row -> {len(input_rows)} rows")

---
## 3 · Identities and Voice Cloner

Every name in `IDENTITIES` needs its own `REFERENCE_<IDENTITY>` WAV; a name
with no reference stops the notebook here rather than converting toward a
guessed default. `voiceclonnx`'s engines differ in whether they take an
`onnxruntime` provider list at all, and in which of their own options they
accept at all — this cell reads the chosen engine's own adapter signature
rather than assuming either, and refuses a `VC_<OPTION>` the engine does not
accept, naming the engine and the option (see the option tables above).

In [ ]:
import inspect

from voiceclonnx import VoiceCloner
from voiceclonnx.engines.base import get_engine

# option name -> typed-parsing kind; the env var is VC_<OPTION UPPERCASE>
VC_OPTION_TYPES = {
    "quantized":     "bool",
    "exaggeration":  "float",
    "ode_steps":     "int",
    "chunk_samples": "int",
    "k":             "int",
    "default_model": "str",
    "sample_rate":   "int",
    "speaker_id":    "int",
    "f0_up_key":     "int",
}


def _parse_typed_env(env_name, raw, kind):
    if kind == "bool":
        low = raw.strip().lower()
        if low in ("1", "true", "yes", "on"):
            return True
        if low in ("0", "false", "no", "off"):
            return False
        raise ValueError(f"{env_name}={raw!r} is not a valid bool (use true/false)")
    if kind == "int":
        try:
            return int(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid int") from None
    if kind == "float":
        try:
            return float(raw)
        except ValueError:
            raise ValueError(f"{env_name}={raw!r} is not a valid float") from None
    if kind == "json":
        try:
            return json.loads(raw)
        except json.JSONDecodeError as exc:
            raise ValueError(f"{env_name}={raw!r} is not valid JSON: {exc}") from None
    return raw


def _resolve_onnx_providers(raw):
    """A comma list, or "auto": CUDA, ROCm, MIGraphX, CPU, CPU always last (docs/pipeline.md)."""
    raw = (raw or "").strip()
    if not raw or raw.lower() == "auto":
        import onnxruntime as ort

        preferred = [
            "CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider",
            "CPUExecutionProvider",
        ]
        available = ort.get_available_providers()
        resolved = [p for p in preferred if p in available]
        if "CPUExecutionProvider" not in resolved:
            resolved.append("CPUExecutionProvider")
        return resolved
    return [p.strip() for p in raw.split(",") if p.strip()]


def _engine_accepted_params(engine_alias):
    """Constructor parameter names of the engine's adapter, minus self and **cfg."""
    adapter_cls = get_engine(engine_alias).adapter_class
    try:
        sig = inspect.signature(adapter_cls.__init__)
    except (TypeError, ValueError):
        return set()
    return {
        name for name, param in sig.parameters.items()
        if name != "self" and param.kind != inspect.Parameter.VAR_KEYWORD
    }


def _engine_accepts_providers(engine_alias):
    """True only for the voiceclonnx engines whose adapter takes a `providers` argument."""
    return "providers" in _engine_accepted_params(engine_alias)


def _resolve_engine_kwargs(engine_alias):
    """Build the VoiceCloner kwargs for `engine_alias` from every VC_<OPTION> env var
    that is set, refusing one the engine's own constructor does not accept."""
    accepted = _engine_accepted_params(engine_alias)
    kwargs = {}
    for option, kind in VC_OPTION_TYPES.items():
        env_name = f"VC_{option.upper()}"
        raw = os.environ.get(env_name, "").strip()
        if not raw:
            continue
        if option not in accepted:
            raise RuntimeError(
                f"{env_name} is set but engine {engine_alias!r} does not accept an option "
                f"named {option!r}. Its constructor accepts: {sorted(accepted) or '(none)'}."
            )
        kwargs[option] = _parse_typed_env(env_name, raw, kind)
    return kwargs


def _resolve_references(identities):
    refs = {}
    for identity in identities:
        env_name = f"REFERENCE_{identity.upper()}"
        path = os.environ.get(env_name, "").strip()
        if not path:
            raise RuntimeError(
                f"{env_name} is not set. Every name in IDENTITIES needs a reference WAV: "
                f"set {env_name}=/path/to/reference.wav and re-run this cell."
            )
        if not os.path.isfile(path):
            raise RuntimeError(f"{env_name}={path!r} does not name a file.")
        refs[identity] = path
    return refs


identities = [i.strip() for i in _env("IDENTITIES", "miro,dii").split(",") if i.strip()]
if not identities:
    raise RuntimeError("IDENTITIES is empty: name at least one voice identity to convert to.")

references = _resolve_references(identities)

vc_engine = _env("VC_ENGINE", "chatterbox")
engine_kwargs = _resolve_engine_kwargs(vc_engine)
if _engine_accepts_providers(vc_engine):
    engine_kwargs["providers"] = _resolve_onnx_providers(_env("ONNX_PROVIDERS", "auto"))

cloner = VoiceCloner(engine=vc_engine, **engine_kwargs)
print(f"Voice cloner ready: engine={cloner.engine}  sample_rate={cloner.sample_rate} Hz")
print(f"  options: {engine_kwargs or '(engine defaults)'}")
for identity, ref_path in references.items():
    print(f"  {identity}: reference={ref_path}")

---
## 4 · Voice Conversion

Converts every input row to every identity and writes one manifest per
identity, following [manifest.md](../docs/manifest.md): `id` is
`<parent id>-<identity>`, `parent_id` names the source row, `identity` is set,
and `audio`, `duration_s` and `sample_rate` describe the converted file at the
cloner's own sample rate. `text`, `lang`, `source` and `speaker` are copied
from the parent row unchanged. Neither this notebook nor voiceclonnx deletes
or moves the source audio.

A row whose output WAV already exists with a nonzero size is left alone
(resume-safe); the manifest for that identity is always rewritten in full, so
its row count matches the input row count on every run, whether a row was
freshly converted or skipped. Writing zero rows for an identity is a failure,
never a silently empty output.

In [ ]:
import os
import hashlib
import time
from importlib.metadata import version as _pkg_version

import soundfile as sf


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def convert_manifest_to_identity(rows, manifest_dir, identity, reference_path, cloner, output_dir, engine, engine_options):
    """Convert every row to `identity`; write its manifest and `.meta.json` sidecar."""
    out_dir = Path(output_dir) / identity
    wavs_dir = out_dir / "wavs"
    wavs_dir.mkdir(parents=True, exist_ok=True)

    out_rows = []
    converted = skipped = 0
    for row in rows:
        out_id = f"{row['id']}-{identity}"
        out_audio_rel = f"wavs/{out_id}.wav"
        out_audio_abs = out_dir / out_audio_rel
        src_audio_abs = Path(manifest_dir) / row["audio"]

        if out_audio_abs.exists() and out_audio_abs.stat().st_size > 0:
            skipped += 1
        else:
            cloner.clone_voice(str(src_audio_abs), reference_path, str(out_audio_abs))
            converted += 1

        info = sf.info(str(out_audio_abs))
        out_rows.append({
            "id": out_id,
            "parent_id": row["id"],
            "identity": identity,
            "audio": out_audio_rel,
            "duration_s": round(info.duration, 3),
            "sample_rate": info.samplerate,
            "text": row["text"],
            "lang": row["lang"],
            "source": row["source"],
            "speaker": row.get("speaker"),
        })

    manifest_out_path = out_dir / f"{identity}.jsonl"
    with open(manifest_out_path, "w", encoding="utf-8") as f:
        for r in out_rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    meta = {
        "notebooks": ["tts_voice_convert"],
        "instruments": {
            "voiceclonnx": {
                "package": "voiceclonnx",
                "version": _pkg_version("voiceclonnx"),
                "engine": engine,
                "options": engine_options,
            },
        },
        "reference": {
            "identity": identity,
            "path": os.path.basename(reference_path),
            "sha256": _sha256(reference_path),
        },
    }
    with open(str(manifest_out_path) + ".meta.json", "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    # Hard check: one output row per input row, per identity.
    assert len(out_rows) == len(rows), (
        f"{identity}: wrote {len(out_rows)} output rows for {len(rows)} input rows"
    )
    if len(out_rows) == 0:
        raise RuntimeError(f"{identity}: zero rows written")

    print(f"  {identity}: {converted} converted, {skipped} skipped (already existed), "
          f"{len(out_rows)} rows -> {manifest_out_path}")
    return out_rows, skipped


CONVERTED = {}
start = time.time()
for identity in identities:
    print(f"Converting to {identity}...")
    out_rows, skipped = convert_manifest_to_identity(
        input_rows, manifest_dir, identity, references[identity], cloner, _env("VC_OUTPUT_DIR"), vc_engine, engine_kwargs
    )
    CONVERTED[identity] = out_rows

elapsed = time.time() - start
print(f"\n{sum(len(v) for v in CONVERTED.values())} rows written across {len(CONVERTED)} identities in {elapsed:.1f}s")

---
## 5 · Output Summary

Confirms, for every identity, that the output manifest holds exactly one row
per input row.

In [ ]:
print("=" * 65)
print("  OUTPUT SUMMARY")
print("=" * 65)
for identity, out_rows in CONVERTED.items():
    manifest_out_path = Path(_env("VC_OUTPUT_DIR")) / identity / f"{identity}.jsonl"
    assert len(out_rows) == len(input_rows), (
        f"{identity}: {len(out_rows)} output rows != {len(input_rows)} input rows"
    )
    print(f"  {identity:>8}: {len(out_rows)} rows  ->  {manifest_out_path}")
print("=" * 65)

---
Licensed under Apache-2.0.
